# 약물 탐지 2-Stage 파이프라인 튜토리얼

**목적**: YOLO26n 기반 약물 탐지(Stage 1) + 분류(Stage 2) 파이프라인 전체 과정을 학습하고 실행

**대상**: 팀 신입, ML 초보자 ~ 데이터 사이언티스트  
**소요 시간**: 약 30분 (GPU 환경 기준)  
**사전 요구사항**: `pip install -r requirements.txt`, GPU 환경 (권장)

---

## 📌 파이프라인 개요

### 왜 2-Stage 파이프라인인가?

1장의 약물 이미지에는 **여러 종류의 알약**이 섞여 있을 수 있습니다.
- **Stage 1 (탐지)**: "어디에 알약이 있는가?" → 각 알약의 위치(bbox) 찾기
- **Stage 2 (분류)**: "이것이 무엇인가?" → 찾은 각 알약의 약품명 분류

### 전체 흐름

```
원본 이미지
    ↓
[Stage 1: YOLO26n 탐지]
  · 학습: 약물 위치 annotation으로 학습
  · 추론: 이미지 → bbox 리스트
    ↓
[Stage 2: ResNet50 분류]
  · 학습: bbox를 crop한 이미지 + 약품명 label로 학습
  · 추론: 각 crop → 약품명 예측
    ↓
최종 결과: [약물위치, 약품명, 신뢰도]
```

### 입출력 형식

| 단계 | 입력 | 출력 | 형식 |
|------|------|------|------|
| **Stage 1 학습** | `data/processed/images/train/*.jpg` + YOLO format labels | 모델 가중치 | `.pt` |
| **Stage 1 추론** | `data/processed/images/val/*.jpg` | `predictions.json` | `{"image_id", "detections": [{bbox, score}]}` |
| **Stage 2 학습** | Crop 이미지 + 분류 label | 분류기 가중치 | `.pt` |
| **Stage 2 추론** | Crop 이미지 | `predictions.json` | `{"crop_id", "class_name", "score"}` |

---

## ⚙️ 1. 환경 설정

먼저 작업 디렉토리와 필수 라이브러리를 로드합니다.


In [ ]:
import os
from pathlib import Path

# 프로젝트 루트 설정
_cwd = Path.cwd()
PROJECT_ROOT = _cwd if (_cwd / "experiments").exists() else _cwd.parent
os.chdir(PROJECT_ROOT)

print(f"✓ 작업 디렉토리: {os.getcwd()}")

In [ ]:
# 시각화·분석용 라이브러리
try:
    import koreanize_matplotlib  # noqa: F401
except ImportError:
    import subprocess

    subprocess.run(["pip", "install", "-q", "koreanize-matplotlib"])
    import koreanize_matplotlib  # noqa: F401, E402

import json
import random
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import yaml
from PIL import Image

print("✓ 모든 라이브러리 로드 완료")

## 2️⃣ 경로 및 설정 확인

학습과 추론에 사용할 설정 파일, 데이터 경로, 출력 디렉토리를 정의합니다.


In [ ]:
# ============ 경로 변수 (수정하지 말 것) ============
# 실험 이름
EXP_NAME = "exp_20260420_baseline_yolo26n"

# 설정 파일 경로
S1_CONFIG = f"experiments/{EXP_NAME}/s1_config.yaml"  # Stage 1 설정
S2_CONFIG = "experiments/stage2_classifier/config.yaml"  # Stage 2 설정
DATA_YAML = "data/processed/dataset.yaml"  # 데이터셋 정의

# 모델 가중치 경로
BEST_PT_S1 = f"experiments/{EXP_NAME}/weights/best.pt"  # Stage 1 best 모델
BEST_PT_S2 = "experiments/stage2_classifier/exp_YYYYMMDD_resnet50_baseline/weights/best.pt"  # Stage 2 best 모델

# 입출력 경로
PRED_OUTPUT = f"experiments/{EXP_NAME}/val_predictions.json"  # Stage 1 추론 결과
CROP_DIR = Path(f"experiments/{EXP_NAME}/stage1_crops")  # Stage 1 추론 crop (시각화용)
GT_CROP_DIR = Path("data/processed/crops")  # GT crop (Stage 2 학습용)
S2_PRED_OUTPUT = f"experiments/{EXP_NAME}/stage2_predictions.json"  # Stage 2 추론 결과

# ============ 경로 존재 확인 ============
paths_to_check = {
    "S1 설정": S1_CONFIG,
    "S2 설정": S2_CONFIG,
    "데이터": DATA_YAML,
    "GT crop": str(GT_CROP_DIR),
}

print("\n경로 확인:")
print("-" * 50)
for name, path in paths_to_check.items():
    exists = "✓" if Path(path).exists() else "✗"
    print(f"{exists} {name:15s}: {path}")
print("-" * 50)

### 설정 파일 확인

**Stage 1 설정** — YOLO26n 모델 학습 파라미터:


In [ ]:
with open(S1_CONFIG) as f:
    cfg = yaml.safe_load(f)

# 핵심 설정만 출력
print("\n📋 Stage 1 주요 설정:")
print("-" * 50)
print(f"  모델: {cfg['model']['name']}")
print(f"  이미지 크기: {cfg['data']['imgsz']}x{cfg['data']['imgsz']}")
print(f"  에포크: {cfg['train']['epochs']}")
print(f"  배치 크기: {cfg['train']['batch']}")
print(f"  Seed: {cfg['seed']} (재현성 보장)")
print("-" * 50)

### 데이터셋 확인

YOLO 형식 데이터셋 구조:
- **Train**: 학습용 이미지 + YOLO format label
- **Val**: 검증용 이미지 (파이프라인 시각화용)
- **Test**: 최종 테스트 (kaggle 제출용)


In [ ]:
with open(DATA_YAML) as f:
    data_cfg = yaml.safe_load(f)

DATA_YAML_DIR = Path(DATA_YAML).parent

print("\n📊 데이터셋 구성:")
print("-" * 50)
for split in ["train", "val", "test"]:
    p = (DATA_YAML_DIR / data_cfg[split]).resolve()
    count = (
        len(list(p.glob("*.jpg"))) + len(list(p.glob("*.png"))) if p.exists() else -1
    )
    status = "✓" if p.exists() else "✗"
    print(f"{status} {split:6s}: {count:4d} images  →  {p}")

VAL_IMG_DIR = (DATA_YAML_DIR / data_cfg["val"]).resolve()
print("-" * 50)

---

## 🎯 Stage 1: 약물 탐지 모델 학습

### Step 1-1: 모델 학습

YOLO26n을 약물 탐지 데이터로 학습합니다.

**입력**: `data/processed/images/train/` (학습 이미지)  
**출력**: `experiments/{EXP_NAME}/weights/best.pt` (최고 성능 모델)

실행 시간: GPU 기준 ~10분

```bash
# 코드 해설:
# - train.py: 학습 스크립트
# - --config: 위에서 확인한 설정 파일
# - --data: 데이터셋 yaml 파일
```


In [ ]:
# Stage 1 학습 실행
print("\n🚀 Stage 1 학습 시작...")
print("=" * 50)
!python scripts/train.py \
    --config {S1_CONFIG} \
    --data   {DATA_YAML}
print("=" * 50)
print("✓ Stage 1 학습 완료\n")

### Step 1-2: 모델 검증

학습된 모델을 검증 데이터에서 평가합니다.

**메트릭**:
- **Precision (P)**: 탐지한 것 중 정답 비율
- **Recall (R)**: 정답 중 탐지한 비율
- **mAP50**: IoU 0.5 기준 평균 정밀도 (일반적)
- **mAP50-95**: IoU 0.5~0.95 기준 평균 정밀도 (엄격함)

목표: mAP50 > 0.95, mAP50-95 > 0.90


In [ ]:
# Stage 1 검증 실행
print("\n🔍 Stage 1 검증 실행...")
print("=" * 50)
!python scripts/validate.py \
    --config  {S1_CONFIG} \
    --data    {DATA_YAML}
print("=" * 50)
print("✓ Stage 1 검증 완료\n")

### Step 1-3: 학습 곡선 시각화

Loss와 mAP 추이를 확인하여 모델 수렴성을 판단합니다.


In [ ]:
import pandas as pd

RESULTS_CSV = Path(f"experiments/{EXP_NAME}/results.csv")

if not RESULTS_CSV.exists():
    print("⚠️  results.csv 없음 — 학습 후 재실행하세요")
else:
    df = pd.read_csv(RESULTS_CSV)
    df.columns = df.columns.str.strip()

    loss_cols = [c for c in df.columns if "loss" in c.lower()]
    map_cols = [c for c in df.columns if "map" in c.lower()]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Loss 곡선
    for col in loss_cols:
        axes[0].plot(df["epoch"], df[col], label=col, linewidth=2)
    axes[0].set_title("손실(Loss) 추이", fontsize=12, fontweight="bold")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # mAP 곡선
    for col in map_cols:
        axes[1].plot(df["epoch"], df[col], label=col, linewidth=2)
    axes[1].set_title("mAP 추이", fontsize=12, fontweight="bold")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("mAP")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(
        f"experiments/{EXP_NAME}/training_curves.png", dpi=150, bbox_inches="tight"
    )
    plt.show()

    print("\n✓ 학습 곡선 저장: experiments/{EXP_NAME}/training_curves.png")

### Step 1-4: 추론 실행

학습된 모델로 검증 데이터에 대해 추론을 수행합니다.

**출력**: `val_predictions.json`
```json
{
  "image_id": "...",
  "detections": [
    {"bbox": [x1, y1, x2, y2], "score": 0.95},
    ...
  ]
}
```

이 결과를 Stage 2 입력으로 사용합니다.


In [ ]:
print("\n🔮 Stage 1 추론 실행...")
print("=" * 50)
!python scripts/predict.py \
    --config  {S1_CONFIG} \
    --source  {VAL_IMG_DIR} \
    --output  {PRED_OUTPUT}
print("=" * 50)

# 결과 통계
with open(PRED_OUTPUT) as f:
    predictions = json.load(f)

total_det = sum(len(p["detections"]) for p in predictions)
avg_det = total_det / max(len(predictions), 1)

print("\n📊 추론 결과 통계:")
print(f"  이미지 수: {len(predictions)}장")
print(f"  총 탐지: {total_det}개")
print(f"  평균 탐지: {avg_det:.2f}개/장")
print()

### Step 1-5: 결과 시각화 — 멀티 탐지 케이스

GT(초록) vs 예측(빨강) 박스를 비교합니다. 알약 2개 이상인 이미지에서 모델 성능을 확인합니다.


In [ ]:
VAL_LABEL_DIR = Path("data/processed/labels/val")


# GT 기준 알약 2개 이상인 이미지만 필터링
def _gt_count(image_id):
    p = VAL_LABEL_DIR / f"{image_id}.txt"
    return len(p.read_text().strip().splitlines()) if p.exists() else 0


multi_det = [p for p in predictions if _gt_count(p["image_id"]) >= 2]
print(f"📊 멀티 탐지 분석: {len(multi_det)}개 / {len(predictions)}개 이미지")

samples_s1 = random.sample(multi_det, min(4, len(multi_det)))

if not samples_s1:
    print("⚠️  GT 기준 알약 2개 이상인 이미지가 없습니다")
else:
    fig, axes = plt.subplots(len(samples_s1), 2, figsize=(14, 5 * len(samples_s1)))
    if len(samples_s1) == 1:
        axes = axes[np.newaxis, :]

    for row, sample in enumerate(samples_s1):
        candidates = list(VAL_IMG_DIR.glob(f"{sample['image_id']}.*"))
        if not candidates:
            axes[row, 0].axis("off")
            axes[row, 1].axis("off")
            continue

        img = np.array(Image.open(candidates[0]).convert("RGB"))
        H, W = img.shape[:2]

        # GT 박스
        label_path = VAL_LABEL_DIR / f"{sample['image_id']}.txt"
        gt_boxes = []
        if label_path.exists():
            for line in label_path.read_text().strip().splitlines():
                _, cx, cy, bw, bh = map(float, line.split())
                x1 = (cx - bw / 2) * W
                y1 = (cy - bh / 2) * H
                x2 = (cx + bw / 2) * W
                y2 = (cy + bh / 2) * H
                gt_boxes.append((x1, y1, x2, y2))

        # GT 시각화
        axes[row, 0].imshow(img)
        for x1, y1, x2, y2 in gt_boxes:
            axes[row, 0].add_patch(
                patches.Rectangle(
                    (x1, y1),
                    x2 - x1,
                    y2 - y1,
                    linewidth=2,
                    edgecolor="green",
                    facecolor="none",
                )
            )
        axes[row, 0].set_title(
            f"GT ({len(gt_boxes)}개)", fontsize=10, fontweight="bold"
        )
        axes[row, 0].axis("off")

        # 예측 시각화
        axes[row, 1].imshow(img)
        for i, det in enumerate(sample["detections"]):
            x1, y1, x2, y2 = det["bbox"]
            axes[row, 1].add_patch(
                patches.Rectangle(
                    (x1, y1),
                    x2 - x1,
                    y2 - y1,
                    linewidth=2,
                    edgecolor="red",
                    facecolor="none",
                )
            )
            axes[row, 1].text(
                x1,
                y1 - 4,
                f"{det['score']:.2f}",
                color="red",
                fontsize=8,
                fontweight="bold",
                bbox=dict(facecolor="red", alpha=0.3, pad=1, edgecolor="none"),
            )
        axes[row, 1].set_title(
            f"예측 ({len(sample['detections'])}개)", fontsize=10, fontweight="bold"
        )
        axes[row, 1].axis("off")

    plt.suptitle(
        "Stage 1 성능 확인 — GT(초록) vs 예측(빨강)", fontsize=13, fontweight="bold"
    )
    plt.tight_layout()
    plt.show()

### Step 1-6: Crop 추출

Stage 1 추론 결과의 bbox를 기준으로 이미지를 잘라냅니다. 이 crop은 Stage 2 입력으로 사용됩니다.

**입력**: `val_predictions.json`, 원본 이미지  
**출력**: `stage1_crops/` (개별 crop 이미지들), `crops_manifest.json` (메타데이터)


In [ ]:
print("\n✂️  Crop 추출 중...")
print("=" * 50)
!python scripts/pipeline/crop.py \
    --predictions {PRED_OUTPUT} \
    --source      {VAL_IMG_DIR} \
    --output      {CROP_DIR} \
    --padding     0.05
print("=" * 50)

# Crop 통계
with open(CROP_DIR / "crops_manifest.json") as f:
    manifest = json.load(f)

print(f"\n✓ Crop 추출 완료: {len(manifest)}개 이미지")

### Step 1-7: Crop 품질 확인

탐지한 알약들이 제대로 crop되었는지 시각적으로 확인합니다.


In [ ]:
# 탐지 수 가장 많은 이미지 선택
manifest_by_img = {}
for item in manifest:
    manifest_by_img.setdefault(item["image_id"], []).append(item)

showcase_id = max(manifest_by_img, key=lambda k: len(manifest_by_img[k]))
showcase_crops = manifest_by_img[showcase_id]
n = len(showcase_crops)

print(f"Showcase 이미지: {showcase_id} ({n}개 탐지)")

orig_img = np.array(
    Image.open(next(VAL_IMG_DIR.glob(f"{showcase_id}.*"))).convert("RGB")
)

fig, axes = plt.subplots(1, n + 1, figsize=(3 * (n + 1), 4))

# 원본 + bbox 번호 표시
axes[0].imshow(orig_img)
for i, item in enumerate(showcase_crops):
    color = plt.cm.tab10.colors[i % 10]
    x1, y1, x2, y2 = item["bbox"]
    axes[0].add_patch(
        patches.Rectangle(
            (x1, y1), x2 - x1, y2 - y1, linewidth=2, edgecolor=color, facecolor="none"
        )
    )
    axes[0].text(x1, y1 - 5, str(i), color=color, fontsize=9, fontweight="bold")
axes[0].set_title("원본 + bbox 번호", fontsize=9)
axes[0].axis("off")

# 각 crop
for i, item in enumerate(showcase_crops):
    color = plt.cm.tab10.colors[i % 10]
    axes[i + 1].imshow(Image.open(CROP_DIR / f"{item['crop_id']}.jpg").convert("RGB"))
    axes[i + 1].set_title(
        f"crop{i}\nconf={item['score']:.2f}", fontsize=8, color=color, fontweight="bold"
    )
    axes[i + 1].axis("off")

plt.suptitle("Stage 1 → Crop 결과 샘플", fontsize=11, fontweight="bold")
plt.tight_layout()
plt.show()

---

## 🎯 Stage 2: 약물 분류 모델

### Step 2-0: GT Crop 생성 (최초 1회)

Stage 2 학습을 위해 GT label 기반 crop을 생성합니다. 이미 존재하면 건너뜁니다.

**출력**: `data/processed/crops/{train,val}/`


In [ ]:
print("\n📦 GT Crop 생성 (또는 로드)...")
print("=" * 50)

if not (GT_CROP_DIR / "train" / "crops_manifest.json").exists():
    print("⏳ 첫 실행 — GT crop 생성 중... (1-2분)")
    !python scripts/pipeline/crop.py \
        --imagefolder data/processed/crops \
        --splits      train val
else:
    import json
    with open(GT_CROP_DIR / "train" / "crops_manifest.json") as f:
        gt_manifest = json.load(f)
    print("✓ GT crop 이미 존재")
    print(f"  Train: {len(gt_manifest)}개 crop")
    
    with open(GT_CROP_DIR / "val" / "crops_manifest.json") as f:
        gt_manifest_val = json.load(f)
    print(f"  Val: {len(gt_manifest_val)}개 crop")

print("=" * 50)

### Step 2-1: 분류 모델 학습

GT crop 이미지들로 약품명 분류 모델(ResNet50)을 학습합니다.

**입력**: `data/processed/crops/{train,val}/` (crop 이미지 + 폴더명 = class)  
**출력**: `experiments/stage2_classifier/weights/best.pt`

학습 시간: GPU 기준 ~10분


In [ ]:
print("\n🚀 Stage 2 분류기 학습 시작...")
print("=" * 50)
!python scripts/pipeline/stage2_train.py \
    --config {S2_CONFIG}
print("=" * 50)
print("✓ Stage 2 학습 완료\n")

### Step 2-2: 분류 추론

Stage 1에서 추출한 crop 이미지들을 분류합니다.

**입력**: `stage1_crops/*.jpg`  
**출력**: `stage2_predictions.json`
```json
{
  "crop_id": "...",
  "class_name": "아스피린",
  "score": 0.92
}
```


In [ ]:
print("\n🔮 Stage 2 추론 실행...")
print("=" * 50)

if not Path(BEST_PT_S2).exists():
    print("⚠️  Stage 2 모델 가중치 없음 — Step 2-1을 먼저 실행하세요")
else:
    !python scripts/pipeline/stage2_predict.py \
        --config  {S2_CONFIG} \
        --weights {BEST_PT_S2} \
        --source  {CROP_DIR} \
        --output  {S2_PRED_OUTPUT}

print("=" * 50)
print()

---

## 📊 2-Stage 통합 결과 시각화

### 최종 결과: 약물 위치 + 약품명 + 신뢰도


In [ ]:
if not Path(S2_PRED_OUTPUT).exists():
    print("⚠️  Stage 2 추론 결과 없음 — Step 2-2를 먼저 실행하세요")
else:
    with open(S2_PRED_OUTPUT) as f:
        s2_results = json.load(f)

    s2_by_crop = {r["crop_id"]: r for r in s2_results}

    # 각 이미지별로 통합 결과 구성
    pipeline_by_img = {}
    for item in manifest:
        s2 = s2_by_crop.get(item["crop_id"], {})
        pipeline_by_img.setdefault(item["image_id"], []).append(
            {
                "bbox": item["bbox"],
                "det_score": item["score"],  # Stage 1 신뢰도
                "crop_id": item["crop_id"],
                "class_name": s2.get("class_name", "?"),  # Stage 2 결과
                "class_score": s2.get("score", 0.0),  # Stage 2 신뢰도
            }
        )

    print(f"\n✓ 통합 결과 준비 완료: {len(pipeline_by_img)}장")

### 결과 1: 오버레이 시각화 — 멀티 탐지 케이스

약물 위치에 약품명을 직접 표시합니다.


In [ ]:
if not Path(S2_PRED_OUTPUT).exists():
    print("⚠️  Stage 2 결과 없음")
else:
    multi_pipeline = {
        img_id: items for img_id, items in pipeline_by_img.items() if len(items) >= 2
    }

    print(f"📊 멀티 탐지 이미지: {len(multi_pipeline)}개 / {len(pipeline_by_img)}개")

    if not multi_pipeline:
        print("⚠️  탐지 2개 이상인 이미지가 없습니다")
    else:
        sample_ids = random.sample(
            list(multi_pipeline.keys()), min(4, len(multi_pipeline))
        )

        fig, axes = plt.subplots(2, 2, figsize=(16, 12))
        for ax, img_id in zip(axes.flatten(), sample_ids):
            candidates = list(VAL_IMG_DIR.glob(f"{img_id}.*"))
            if not candidates:
                ax.axis("off")
                continue

            img = np.array(Image.open(candidates[0]).convert("RGB"))
            ax.imshow(img)

            for i, item in enumerate(multi_pipeline[img_id]):
                x1, y1, x2, y2 = item["bbox"]
                color = plt.cm.tab10.colors[i % 10]

                # bbox
                ax.add_patch(
                    patches.Rectangle(
                        (x1, y1),
                        x2 - x1,
                        y2 - y1,
                        linewidth=2.5,
                        edgecolor=color,
                        facecolor="none",
                    )
                )

                # 약품명 + 신뢰도
                label_text = f"{item['class_name']}\nS1:{item['det_score']:.2f} S2:{item['class_score']:.2f}"
                ax.text(
                    x1,
                    y2 + 4,
                    label_text,
                    color="white",
                    fontsize=8,
                    fontweight="bold",
                    bbox=dict(
                        facecolor=color,
                        alpha=0.8,
                        pad=2,
                        edgecolor="white",
                        linewidth=1,
                    ),
                )

            n = len(multi_pipeline[img_id])
            ax.set_title(f"{img_id}  ({n}개 약물 탐지)", fontsize=10, fontweight="bold")
            ax.axis("off")

        for ax in axes.flatten()[len(sample_ids) :]:
            ax.axis("off")

        plt.suptitle(
            "🎯 최종 결과: 약물 위치(bbox) + 약품명 + 신뢰도\n"
            "S1: Stage 1 탐지 신뢰도 | S2: Stage 2 분류 신뢰도",
            fontsize=12,
            fontweight="bold",
        )
        plt.tight_layout()
        plt.savefig(
            f"experiments/{EXP_NAME}/pipeline_overlay.png", dpi=150, bbox_inches="tight"
        )
        plt.show()

---

## 💡 결과 해석 및 다음 단계

### 성능 지표 해석

| 메트릭 | 의미 | 목표값 |
|--------|------|--------|
| **S1 Precision** | 탐지한 것 중 정답 비율 | > 0.95 |
| **S1 Recall** | 정답 중 탐지한 비율 | > 0.95 |
| **S1 mAP50** | 탐지 정확도 (관대함) | > 0.95 |
| **S2 Top-1 Accuracy** | 분류 정확도 (1순위만 맞음) | > 0.80 |
| **S2 Top-5 Accuracy** | 분류 정확도 (상위 5개 포함) | > 0.95 |

### 개선 아이디어

1. **데이터 증강 강화**: albumentations 설정에서 `mosaic`, `mixup` 비율 증가
2. **모델 업그레이드**: `yolo26n` → `yolo26m` (더 큰 모델)
3. **앙상블**: 여러 모델 결과를 앙상블해서 신뢰도 향상
4. **임계값 조정**: 신뢰도 threshold를 조정해 precision/recall 트레이드오프 최적화
5. **하드 샘플 마이닝**: 자주 틀리는 이미지들을 수집해 재학습

### 다음 단계

- ✅ 파이프라인 이해 완료
- 📊 다른 config로 실험 ([참고 가이드](./experiments/README.md))
- 🔧 Kaggle 제출 준비 ([제출 가이드](../scripts/README.md#kaggle-submission))
- 📈 결과 분석 및 보고서 작성

---

### 📚 참고 자료

- **스크립트**: `scripts/` 디렉토리
- **설정**: `experiments/*/config.yaml`
- **데이터**: `data/processed/dataset.yaml`

문제 발생 시: 팀 Slack 또는 이슈 등록